# `LinearRegression` and Ordinary Least Squares

**Part 1 — `list[dict]` in, `list[dict]` out**
**Part 2 — `DataFrame` in, `DataFrame` out**

---

The companion notebook trained a linear model by **stochastic gradient descent**: start at zero,
look at one sample, nudge the weights, repeat a few thousand times.

`LinearRegression` reaches the same kind of model by a completely different route. There is no
learning rate, no epochs, no shuffling, no convergence check. Instead we **solve for the best
weights directly**, in one shot, with linear algebra. That method is **Ordinary Least Squares**.

| | `SGDRegressor` | `LinearRegression` (OLS) |
|---|---|---|
| How | iterative descent | closed-form solve |
| Hyper-parameters | `eta0`, `power_t`, `max_iter`, `tol`, … | essentially none |
| Feature scaling | **required** | irrelevant to the answer |
| Result | approximate, seed-dependent | exact, deterministic |
| Scales to | huge `n`, streaming data | modest `n_features` |

**Roadmap**

| Step | What we build |
|---|---|
| 1 | The data (same as the SGD notebook, for comparability) |
| 2 | `list[dict]` → matrix, with the same guards |
| 3 | What OLS actually minimises, and the normal equations |
| 4 | Three ways to solve them |
| 5 | Where the intercept comes from |
| 6 | The collinearity trap — three silent failures |
| 7 | Conditioning: why `inv` is the bad one |
| 8 | The `LinearRegressionDict` class |
| 9 | Check against scikit-learn |
| 10 | Scale invariance — the sharpest contrast with SGD |
| 11 | Standard errors, t-statistics, confidence intervals |
| 12 | `DataFrame` in, `DataFrame` out |
| 13 | One core, two adapters |
| 14 | OLS vs SGD, measured |

In [1]:
import numpy as np
import pandas as pd

np.set_printoptions(precision=4, suppress=True)
pd.set_option("display.precision", 4)
print("numpy", np.__version__, "| pandas", pd.__version__)

numpy 2.4.4 | pandas 3.0.2


---
## Step 1 — The data

Identical generator and seed to the SGD notebook, so every number here is directly comparable.

$$\text{price}_k = 0.15 \cdot \text{sqft} + 10 \cdot \text{bedrooms} - 0.8 \cdot \text{age} + 50 + \varepsilon,
\qquad \varepsilon \sim N(0, 15^2)$$

In [2]:
rng = np.random.default_rng(42)
n = 300

sqft     = rng.uniform(800, 3500, n).round(0)
bedrooms = rng.integers(1, 6, n)
age      = rng.uniform(0, 60, n).round(1)

TRUE_COEF = {"sqft": 0.15, "bedrooms": 10.0, "age": -0.8}
TRUE_INTERCEPT = 50.0
NOISE_SD = 15.0

price_k = (TRUE_COEF["sqft"] * sqft
           + TRUE_COEF["bedrooms"] * bedrooms
           + TRUE_COEF["age"] * age
           + TRUE_INTERCEPT
           + rng.normal(0, NOISE_SD, n))

records = [{"sqft": float(s), "bedrooms": int(b), "age": float(a)}
           for s, b, a in zip(sqft, bedrooms, age)]
targets = price_k.tolist()

idx = rng.permutation(n)
cut = int(0.8 * n)
train_i, test_i = idx[:cut], idx[cut:]

X_train = [records[i] for i in train_i]
y_train = [targets[i] for i in train_i]
X_test  = [records[i] for i in test_i]
y_test  = [targets[i] for i in test_i]

print(f"train: {len(X_train)} rows | test: {len(X_test)} rows")
print("first record:", X_train[0], "-> target", round(y_train[0], 2))

train: 240 rows | test: 60 rows
first record: {'sqft': 912.0, 'bedrooms': 4, 'age': 51.1} -> target 187.89


---
## Step 2 — `list[dict]` → matrix

Same contract as the SGD notebook, and for the same reasons: dictionaries carry no guaranteed
key order, so the feature order is **frozen at `fit` time** as `sorted(union of keys)`, a
missing key raises rather than silently becoming `0`, and an unknown key raises too.

In [3]:
def extract_feature_names(X):
    names = set()
    for row in X:
        names.update(row.keys())
    return sorted(names)


def records_to_matrix(X, feature_names):
    known = set(feature_names)
    out = np.empty((len(X), len(feature_names)), dtype=float)
    for i, row in enumerate(X):
        missing = [f for f in feature_names if f not in row]
        if missing:
            raise ValueError(
                f"record {i} is missing feature(s) {missing}; expected exactly {feature_names}. "
                f"A missing feature is not a zero -- fill it explicitly if you mean 0."
            )
        unknown = [k for k in row if k not in known]
        if unknown:
            raise ValueError(f"record {i} has unknown feature(s) {unknown}; fitted on {feature_names}")
        out[i] = [row[f] for f in feature_names]
    return out


feature_names = extract_feature_names(X_train)
X_mat = records_to_matrix(X_train, feature_names)
y_arr = np.asarray(y_train, dtype=float)

print("feature order:", feature_names)
print("X shape:", X_mat.shape, "| y shape:", y_arr.shape)
print(X_mat[:3])

feature order: ['age', 'bedrooms', 'sqft']
X shape: (240, 3) | y shape: (240,)
[[  51.1    4.   912. ]
 [  45.1    1.   848. ]
 [  51.8    3.  3297. ]]


---
## Step 3 — What OLS minimises

Same model as before, $\hat{y} = \mathbf{w}\cdot\mathbf{x} + b$. Same loss, too — squared error —
except now we write it over the **whole dataset at once** instead of one sample at a time:

$$\text{RSS}(\boldsymbol\beta) = \sum_{i=1}^{n}(y_i - \mathbf{x}_i \cdot \boldsymbol\beta)^2
= \lVert \mathbf{y} - X\boldsymbol\beta \rVert^2$$

This is a quadratic bowl in $\boldsymbol\beta$. It has exactly one minimum, and at the minimum
the gradient is zero:

$$\nabla_{\boldsymbol\beta}\,\text{RSS} = -2X^{\top}(\mathbf{y} - X\boldsymbol\beta) = \mathbf{0}$$

Rearranged, that is the **normal equations**:

$$\boxed{\;X^{\top}X\,\boldsymbol\beta = X^{\top}\mathbf{y}\;}$$

SGD spends thousands of updates crawling toward this point. OLS just writes it down.

**The geometry.** $X\boldsymbol\beta$ can only ever land in the column space of $X$ — the
subspace spanned by the features. The closest point in that subspace to $\mathbf{y}$ is its
**orthogonal projection**, and "orthogonal" means the residual is perpendicular to every
feature: $X^{\top}(\mathbf{y} - X\boldsymbol\beta) = \mathbf{0}$. That is the same equation
again. We check it numerically below.

In [4]:
def rss(X, y, beta):
    resid = y - X @ beta
    return float(resid @ resid)


# Design matrix: a leading column of ones absorbs the intercept into beta[0].
def add_intercept_column(X):
    return np.column_stack([np.ones(len(X)), X])


D = add_intercept_column(X_mat)
print("design matrix:", D.shape, "-> beta will be [intercept, " + ", ".join(feature_names) + "]")

beta_true = np.array([TRUE_INTERCEPT] + [TRUE_COEF[f] for f in feature_names])
beta_ols, *_ = np.linalg.lstsq(D, y_arr, rcond=None)

print("\nRSS at the TRUE coefficients:", round(rss(D, y_arr, beta_true), 1))
print("RSS at the OLS coefficients :", round(rss(D, y_arr, beta_ols), 1), " <- lower, by construction")

design matrix: (240, 4) -> beta will be [intercept, age, bedrooms, sqft]

RSS at the TRUE coefficients: 55814.6
RSS at the OLS coefficients : 55536.5  <- lower, by construction


The OLS fit beats the *true* coefficients on training RSS. That is not a bug — OLS minimises
error on **this sample**, noise included. It is the cleanest illustration of overfitting you
can get: the minimiser of training error is not the truth.

In [5]:
# The orthogonality condition: residuals perpendicular to every column of the design matrix.
resid = y_arr - D @ beta_ols
print("X^T r  =", (D.T @ resid).round(9))
print("\nFirst entry is the ones-column, so it says the residuals sum to zero:",
      round(float(resid.sum()), 9))
abs_err = np.abs(D.T @ resid).max()
rel_err = abs_err / (np.linalg.norm(D) * np.linalg.norm(resid))
print(f"\nmax |X^T r| = {abs_err:.2e} in absolute terms, which sounds large --")
print(f"but relative to ||X||*||r|| = {np.linalg.norm(D) * np.linalg.norm(resid):.2e} it is {rel_err:.2e}.")
print("Scale-free, that is zero to floating-point precision. Always compare against the scale involved.")

X^T r  = [0. 0. 0. 0.]

First entry is the ones-column, so it says the residuals sum to zero: 0.0

max |X^T r| = 1.03e-07 in absolute terms, which sounds large --
but relative to ||X||*||r|| = 8.29e+06 it is 1.24e-14.
Scale-free, that is zero to floating-point precision. Always compare against the scale involved.


---
## Step 4 — Three ways to solve the normal equations

$X^{\top}X\boldsymbol\beta = X^{\top}\mathbf{y}$ is a linear system. Three candidates:

| Approach | Code | Verdict |
|---|---|---|
| Invert the matrix | `inv(X.T @ X) @ X.T @ y` | **never do this** |
| Solve the system | `solve(X.T @ X, X.T @ y)` | better, still fragile |
| Least-squares directly | `lstsq(X, y)` | **use this** |

On well-behaved data all three agree. Step 6 is where they stop agreeing.

In [6]:
def fit_via_inv(D, y):
    return np.linalg.inv(D.T @ D) @ D.T @ y

def fit_via_solve(D, y):
    return np.linalg.solve(D.T @ D, D.T @ y)

def fit_via_lstsq(D, y):
    beta, residuals, rank, singular = np.linalg.lstsq(D, y, rcond=None)
    return beta


labels = ["intercept"] + feature_names
table = pd.DataFrame(
    {name: fn(D, y_arr) for name, fn in
     [("inv", fit_via_inv), ("solve", fit_via_solve), ("lstsq", fit_via_lstsq)]},
    index=labels,
)
table["true"] = beta_true
print(table)
solvers = table[["inv", "solve", "lstsq"]].to_numpy()
print("\nmax pairwise disagreement:", f"{np.ptp(solvers, axis=1).max():.2e}")

               inv    solve    lstsq   true
intercept  53.1631  53.1631  53.1631  50.00
age        -0.8290  -0.8290  -0.8290  -0.80
bedrooms    9.9675   9.9675   9.9675  10.00
sqft        0.1488   0.1488   0.1488   0.15

max pairwise disagreement: 2.06e-12


---
## Step 5 — Where the intercept comes from

Two equivalent routes:

1. **Augment** — glue a column of ones onto $X$ and let $\beta_0$ be the intercept. What we did
   above. Simple, but the ones-column gets swept into any regularisation you add later.
2. **Centre** — subtract the column means from $X$ and the mean from $y$, solve for the slopes
   on centred data, then recover
   $$b = \bar{y} - \mathbf{w}\cdot\bar{\mathbf{x}}$$
   This is what scikit-learn does. It keeps the intercept out of the penalty term and shrinks
   the system by one dimension.

The identity $b = \bar{y} - \mathbf{w}\cdot\bar{\mathbf{x}}$ says the fitted plane always passes
through the centroid of the data.

In [7]:
def fit_centered(X, y):
    x_bar, y_bar = X.mean(axis=0), y.mean()
    coef, *_ = np.linalg.lstsq(X - x_bar, y - y_bar, rcond=None)
    intercept = y_bar - coef @ x_bar
    return coef, intercept


coef_c, intercept_c = fit_centered(X_mat, y_arr)

print("augmented  :", np.round(beta_ols, 6))
print("centred    :", np.round(np.r_[intercept_c, coef_c], 6))
print("max diff   :", f"{np.abs(np.r_[intercept_c, coef_c] - beta_ols).max():.2e}")

pred_centroid = coef_c @ X_mat.mean(axis=0) + intercept_c
print(f"\nprediction at the centroid = {pred_centroid:.6f}   mean of y = {y_arr.mean():.6f}")

augmented  : [53.1631 -0.829   9.9675  0.1488]
centred    : [53.1631 -0.829   9.9675  0.1488]
max diff   : 4.76e-13

prediction at the centroid = 376.341360   mean of y = 376.341360


---
## Step 6 — The collinearity trap

Suppose someone adds floor area in square metres alongside square feet. It carries **zero new
information**: `sqft_m2 = sqft * 0.092903`. Now the columns of $X$ are linearly dependent,
$X^{\top}X$ is singular, and $\boldsymbol\beta$ is no longer unique — infinitely many
coefficient vectors give the identical fit.

Watch all three solvers here. None of them raises an exception.

In [8]:
SQFT_TO_M2 = 0.092903
X_dup = np.column_stack([X_mat, X_mat[:, feature_names.index("sqft")] * SQFT_TO_M2])
D_dup = add_intercept_column(X_dup)
labels_dup = ["intercept"] + feature_names + ["sqft_m2"]

results = {}
for name, fn in [("inv", fit_via_inv), ("solve", fit_via_solve), ("lstsq", fit_via_lstsq)]:
    try:
        b = fn(D_dup, y_arr)
        results[name] = b
        print(f"{name:>6}: RSS = {rss(D_dup, y_arr, b):>12,.1f}   ||beta|| = {np.linalg.norm(b):>10.3f}")
    except np.linalg.LinAlgError as e:
        print(f"{name:>6}: LinAlgError: {e}")

print()
print(pd.DataFrame(results, index=labels_dup))

   inv: RSS =    323,173.4   ||beta|| =     94.123
 solve: RSS =     55,536.5   ||beta|| =     54.138
 lstsq: RSS =     55,536.5   ||beta|| =     54.096

               inv    solve    lstsq
intercept  93.6613  53.1631  53.1631
age        -0.7582  -0.8290  -0.8290
bedrooms    9.1407   9.9675   9.9675
sqft       -0.0057  -0.0489   0.1476
sqft_m2     1.6288   2.1281   0.0137


Read that table carefully, because each column fails differently:

- **`inv`** is simply **wrong**. Its RSS is several times larger than the others', and it claims
  a *negative* coefficient on `sqft`. It did not crash. It did not warn. It returned confident
  nonsense produced by inverting a matrix that has no inverse.
- **`solve`** achieves the correct minimum RSS, so it is a legitimate answer — but it splits the
  area effect between `sqft` and `sqft_m2` arbitrarily. Perturb the data slightly and the split
  jumps around. The fit is stable; the coefficients are not.
- **`lstsq`** also achieves the minimum RSS, and among all the tied solutions it returns the one
  with the **smallest norm** — a deterministic, reproducible choice. This is what scikit-learn's
  `LinearRegression` returns.

The invariant that survives: the *total* area effect. Individual coefficients are meaningless
here, but their combination is not.

In [9]:
i_sqft = labels_dup.index("sqft")
i_m2 = labels_dup.index("sqft_m2")

print("total effect of +1 sqft  =  coef_sqft + coef_sqft_m2 * 0.092903")
for name, b in results.items():
    print(f"  {name:>6}: {b[i_sqft]:>9.5f} + {b[i_m2]:>9.5f} * {SQFT_TO_M2} = {b[i_sqft] + b[i_m2] * SQFT_TO_M2:.6f}")
print(f"  {'clean':>6}: {beta_ols[labels.index('sqft')]:.6f}   (from the non-duplicated fit)")

total effect of +1 sqft  =  coef_sqft + coef_sqft_m2 * 0.092903
     inv:  -0.00567 +   1.62878 * 0.092903 = 0.145650
   solve:  -0.04887 +   2.12814 * 0.092903 = 0.148836
   lstsq:   0.14756 +   0.01371 * 0.092903 = 0.148836
   clean: 0.148836   (from the non-duplicated fit)


---
## Step 7 — Conditioning: why `inv` is the bad one

The **condition number** measures how much a solution can move when the input is nudged. The
damage done by forming $X^{\top}X$ is exact and brutal:

$$\kappa(X^{\top}X) = \kappa(X)^2$$

Squaring the condition number **halves the number of reliable digits**. `lstsq` works on $X$
directly via an SVD and never forms $X^{\top}X$, which is the whole reason it survives Step 6.

In [10]:
# The identity, checked on a well-conditioned matrix where both numbers are meaningful.
kx, kg = np.linalg.cond(D), np.linalg.cond(D.T @ D)
print(f"clean:  cond(X) = {kx:.3e}   cond(X)^2 = {kx**2:.3e}   cond(X'X) = {kg:.3e}")
print(f"        ratio cond(X'X) / cond(X)^2 = {kg / kx**2:.6f}   <- the identity holds")
print(f"\n        solving via lstsq   -> lose ~log10(cond(X))   = {np.log10(kx):.1f} digits")
print(f"        solving via X'X     -> lose ~log10(cond(X'X)) = {np.log10(kg):.1f} digits")
print(f"        so forming X'X costs an EXTRA {np.log10(kg) - np.log10(kx):.1f} digits, "
      f"out of the ~16 double precision gives us")

# On the rank-deficient matrix the identity cannot be checked at all.
kx_d, kg_d = np.linalg.cond(D_dup), np.linalg.cond(D_dup.T @ D_dup)
print(f"\nduplicated column:  cond(X) = {kx_d:.3e}   cond(X'X) = {kg_d:.3e}")
print(f"        ratio = {kg_d / kx_d**2:.6f}   <- meaningless; see the note below")
print(f"\nrank of the duplicated design matrix: {np.linalg.matrix_rank(D_dup)} "
      f"of {D_dup.shape[1]} columns -> rank deficient")
print(f"smallest singular values: {np.linalg.svd(D_dup, compute_uv=False)[-2:]}")

clean:  cond(X) = 9.298e+03   cond(X)^2 = 8.645e+07   cond(X'X) = 8.645e+07
        ratio cond(X'X) / cond(X)^2 = 1.000000   <- the identity holds

        solving via lstsq   -> lose ~log10(cond(X))   = 4.0 digits
        solving via X'X     -> lose ~log10(cond(X'X)) = 7.9 digits
        so forming X'X costs an EXTRA 4.0 digits, out of the ~16 double precision gives us

duplicated column:  cond(X) = 5.017e+16   cond(X'X) = 9.710e+16
        ratio = 0.000000   <- meaningless; see the note below

rank of the duplicated design matrix: 4 of 5 columns -> rank deficient
smallest singular values: [3.7819 0.    ]


Two things to take from that output.

The identity holds exactly on the clean matrix: 8.645e+07 really is the square of 9.298e+03.

The digit counts are worth reading carefully, because the two numbers measure different things.
A least-squares solve loses roughly $\log_{10}\kappa(X) \approx 4.0$ digits; going through the
normal equations loses roughly $\log_{10}\kappa(X^{\top}X) \approx 7.9$. The **extra** price of
forming $X^{\top}X$ is the difference — about 4 digits. Out of 16 that is survivable here. On a
problem with $\kappa(X) \approx 10^8$ it is the difference between a usable answer and no
correct digits at all.

On the duplicated matrix the ratio comes out as 0.0000, which looks like the identity breaking.
It is not. In exact arithmetic both condition numbers are **infinite**, because the matrix is
genuinely singular. What LAPACK reports instead is determined by rounding noise near 1e-16, so
the two numbers saturate independently and their ratio is meaningless. The honest reading is
"both are astronomically large, and no further digits should be believed" -- which the smallest
singular value confirms.

---
## Step 8 — The class

Same API shape as the SGD version: hyper-parameters in `__init__`, learning in `fit`, learned
attributes ending in `_`, and `predict` returning a `list[dict]`.

We mirror scikit-learn's attributes: `coef_`, `intercept_`, `n_features_in_`, plus `rank_` and
`singular_`, which exist precisely so you can detect the Step 6 situation.

In [11]:
class LinearRegressionDict:
    # Ordinary least squares.  list[dict] in, list[dict] out.

    def __init__(self, fit_intercept=True, solver="lstsq", output_key="prediction"):
        if solver not in {"lstsq", "solve", "inv"}:
            raise ValueError(f"solver must be 'lstsq', 'solve' or 'inv'; got {solver!r}")
        self.fit_intercept = fit_intercept
        self.solver = solver
        self.output_key = output_key

    def fit(self, X, y):
        if len(X) != len(y):
            raise ValueError(f"X has {len(X)} records but y has {len(y)} targets")

        self.feature_names_ = extract_feature_names(X)
        Xm = records_to_matrix(X, self.feature_names_)
        yv = np.asarray(y, dtype=float)

        if not np.isfinite(Xm).all():
            raise ValueError("X contains NaN or inf; clean or impute before fitting")
        if Xm.shape[0] < Xm.shape[1] + int(self.fit_intercept):
            raise ValueError(
                f"{Xm.shape[0]} samples is fewer than the {Xm.shape[1] + int(self.fit_intercept)} "
                f"parameters to estimate; the system is underdetermined"
            )

        self.n_features_in_ = Xm.shape[1]

        # Centre, so the intercept stays out of the linear system (what sklearn does).
        if self.fit_intercept:
            self.x_offset_, self.y_offset_ = Xm.mean(axis=0), yv.mean()
        else:
            self.x_offset_, self.y_offset_ = np.zeros(Xm.shape[1]), 0.0
        Xc, yc = Xm - self.x_offset_, yv - self.y_offset_

        self.rank_ = int(np.linalg.matrix_rank(Xc))
        self.singular_ = np.linalg.svd(Xc, compute_uv=False)
        if self.rank_ < Xc.shape[1] and self.solver != "lstsq":
            raise np.linalg.LinAlgError(
                f"design matrix is rank deficient (rank {self.rank_} < {Xc.shape[1]} columns). "
                f"solver={self.solver!r} would return an arbitrary or numerically corrupt answer; "
                f"use solver='lstsq', or drop the redundant feature."
            )

        if self.solver == "lstsq":
            self.coef_, *_ = np.linalg.lstsq(Xc, yc, rcond=None)
        elif self.solver == "solve":
            self.coef_ = np.linalg.solve(Xc.T @ Xc, Xc.T @ yc)
        else:
            self.coef_ = np.linalg.inv(Xc.T @ Xc) @ Xc.T @ yc

        self.intercept_ = float(self.y_offset_ - self.coef_ @ self.x_offset_)
        return self

    def predict(self, X):
        if not hasattr(self, "coef_"):
            raise RuntimeError("call fit() before predict()")
        scores = records_to_matrix(X, self.feature_names_) @ self.coef_ + self.intercept_
        return [{self.output_key: float(s)} for s in scores]

    def score(self, X, y):
        yv = np.asarray(y, dtype=float)
        pred = np.array([d[self.output_key] for d in self.predict(X)])
        return 1 - np.sum((yv - pred) ** 2) / np.sum((yv - yv.mean()) ** 2)

    @property
    def coef_dict_(self):
        return dict(zip(self.feature_names_, self.coef_))


ols = LinearRegressionDict().fit(X_train, y_train)

print("coef_     :", {k: round(float(v), 4) for k, v in ols.coef_dict_.items()})
print("intercept_:", round(ols.intercept_, 4))
print("rank_     :", ols.rank_, "of", ols.n_features_in_, "features")
print("singular_ :", ols.singular_.round(2))
print(f"\nR^2 train: {ols.score(X_train, y_train):.4f}")
print(f"R^2 test : {ols.score(X_test,  y_test):.4f}")

coef_     : {'age': -0.829, 'bedrooms': 9.9675, 'sqft': 0.1488}
intercept_: 53.1631
rank_     : 3 of 3 features
singular_ : [12146.98   266.2     22.21]

R^2 train: 0.9837
R^2 test : 0.9869


Compare with the SGD notebook: **no standardisation step, no learning rate, no epochs, no
`loss_curve_`.** One call, done. And note the guard we added — with `solver='solve'` or `'inv'`,
rank deficiency now raises instead of returning the nonsense from Step 6.

In [12]:
X_train_dup = [dict(r, sqft_m2=r["sqft"] * SQFT_TO_M2) for r in X_train]

for solver in ["inv", "solve", "lstsq"]:
    try:
        m = LinearRegressionDict(solver=solver).fit(X_train_dup, y_train)
        print(f"{solver:>6}: fitted. rank_={m.rank_}, R^2 train = {m.score(X_train_dup, y_train):.4f}")
    except np.linalg.LinAlgError as e:
        print(f"{solver:>6}: LinAlgError: {str(e)[:96]}...")

   inv: LinAlgError: design matrix is rank deficient (rank 3 < 4 columns). solver='inv' would return an arbitrary or ...
 solve: LinAlgError: design matrix is rank deficient (rank 3 < 4 columns). solver='solve' would return an arbitrary o...
 lstsq: fitted. rank_=3, R^2 train = 0.9837


---
## Step 9 — Check against scikit-learn

For dense inputs, `LinearRegression` hands the problem to an SVD-based least-squares routine
(`scipy.linalg.lstsq`) rather than forming the normal equations — the same strategy as our
`lstsq` path. So unlike the SGD comparison, where we could only expect the same *destination*,
here we should match to floating-point noise.

In [13]:
from sklearn.linear_model import LinearRegression

X_train_arr = records_to_matrix(X_train, feature_names)
X_test_arr  = records_to_matrix(X_test,  feature_names)

sk = LinearRegression().fit(X_train_arr, y_train)

print(pd.DataFrame({"ours": ols.coef_, "sklearn": sk.coef_}, index=feature_names))
print(f"\nintercept   ours={ols.intercept_:.10f}   sklearn={sk.intercept_:.10f}")
print(f"max |coef difference| = {np.abs(ols.coef_ - sk.coef_).max():.2e}")
print(f"|intercept difference| = {abs(ols.intercept_ - sk.intercept_):.2e}")
print(f"\nrank_     ours={ols.rank_}  sklearn={sk.rank_}")
print(f"R^2 test  ours={ols.score(X_test, y_test):.6f}  sklearn={sk.score(X_test_arr, y_test):.6f}")

            ours  sklearn
age      -0.8290  -0.8290
bedrooms  9.9675   9.9675
sqft      0.1488   0.1488

intercept   ours=53.1630798833   sklearn=53.1630798833
max |coef difference| = 0.00e+00
|intercept difference| = 0.00e+00

rank_     ours=3  sklearn=3
R^2 test  ours=0.986884  sklearn=0.986884


In [14]:
# The output contract: list[dict], one per input record.
preds = ols.predict(X_test)
print(type(preds), "of", type(preds[0]), "| length", len(preds), "== inputs", len(X_test))
for p, actual in zip(preds[:5], y_test[:5]):
    print(f"  {{'prediction': {p['prediction']:.2f}}}   actual={actual:.2f}")

<class 'list'> of <class 'dict'> | length 60 == inputs 60
  {'prediction': 287.16}   actual=284.97
  {'prediction': 234.08}   actual=231.18
  {'prediction': 268.67}   actual=270.87
  {'prediction': 191.72}   actual=200.08
  {'prediction': 408.92}   actual=410.79


---
## Step 10 — Scale invariance

In the SGD notebook, raw features made the loss diverge to `nan` within one epoch, and
standardisation was mandatory. OLS does not care at all.

If you rescale a feature by $c$, its coefficient is divided by $c$ and **every prediction stays
identical**. This is not an approximation that holds after enough iterations — it is exact
algebra, visible to machine precision.

In [15]:
scaler_mean = X_train_arr.mean(axis=0)
scaler_std  = X_train_arr.std(axis=0)
X_train_std = [dict(zip(feature_names, row)) for row in (X_train_arr - scaler_mean) / scaler_std]
X_test_std  = [dict(zip(feature_names, row)) for row in (X_test_arr  - scaler_mean) / scaler_std]

ols_raw = LinearRegressionDict().fit(X_train, y_train)
ols_std = LinearRegressionDict().fit(X_train_std, y_train)

print("raw-feature coef       :", ols_raw.coef_.round(6))
print("standardised coef      :", ols_std.coef_.round(6))
print("standardised / scale   :", (ols_std.coef_ / scaler_std).round(6), " <- same as raw")

p_raw = np.array([d["prediction"] for d in ols_raw.predict(X_test)])
p_std = np.array([d["prediction"] for d in ols_std.predict(X_test_std)])
print(f"\nmax |prediction difference| = {np.abs(p_raw - p_std).max():.2e}")
print(f"R^2  raw = {ols_raw.score(X_test, y_test):.10f}")
print(f"R^2  std = {ols_std.score(X_test_std, y_test):.10f}")

raw-feature coef       : [-0.829   9.9675  0.1488]
standardised coef      : [-14.2438  14.2998 116.7002]
standardised / scale   : [-0.829   9.9675  0.1488]  <- same as raw

max |prediction difference| = 5.97e-13
R^2  raw = 0.9868843582
R^2  std = 0.9868843582


So standardising before OLS changes nothing about the fit. It is still worth doing for two
*other* reasons: making coefficients comparable to each other ("effect per standard deviation"),
and improving conditioning when features span wildly different magnitudes. But it is a
readability and numerical-hygiene choice, not a correctness requirement — which is exactly the
opposite of the SGD situation.

---
## Step 11 — What the closed form buys you: uncertainty

`coef_sqft = 0.1484` — but how precise is that? SGD cannot tell you. OLS can, in closed form.

Under the standard assumptions the covariance of the estimates is

$$\widehat{\text{Var}}(\boldsymbol\beta) = \hat{\sigma}^2 (X^{\top}X)^{-1},
\qquad \hat{\sigma}^2 = \frac{\text{RSS}}{n - p}$$

with $p$ the number of estimated parameters, intercept included. The standard errors are the
square roots of the diagonal.

**The assumptions this rests on** — state them, because the formula prints numbers regardless:

1. the relationship really is linear in the parameters;
2. errors are independent across rows;
3. errors have constant variance (homoscedasticity);
4. errors are normal — needed for the $t$-distribution to be exact. With large $n$ the CLT makes
   it approximate rather than exact, which is usually fine.

Our synthetic data satisfies all four by construction, which is precisely why it is the right
place to *check* the formula rather than just trust it.

In [16]:
from scipy import stats


def ols_inference(X, y, feature_names, alpha=0.05):
    # Returns list[dict] -- one dict per parameter, intercept first.
    D = add_intercept_column(X)
    n_obs, p = D.shape
    beta, *_ = np.linalg.lstsq(D, y, rcond=None)

    resid = y - D @ beta
    dof = n_obs - p
    sigma2 = float(resid @ resid) / dof
    cov = sigma2 * np.linalg.inv(D.T @ D)
    se = np.sqrt(np.diag(cov))

    t_stat = beta / se
    p_value = 2 * stats.t.sf(np.abs(t_stat), dof)
    crit = stats.t.ppf(1 - alpha / 2, dof)

    return [
        {"term": name, "coef": float(b), "std_err": float(s), "t": float(t),
         "p_value": float(pv), "ci_low": float(b - crit * s), "ci_high": float(b + crit * s)}
        for name, b, s, t, pv in zip(["intercept"] + feature_names, beta, se, t_stat, p_value)
    ]


stats_rows = ols_inference(X_mat, y_arr, feature_names)
for row in stats_rows:
    print(f"{row['term']:>10}  coef={row['coef']:>9.4f}  se={row['std_err']:>7.4f}  "
          f"t={row['t']:>8.2f}  p={row['p_value']:.2e}  95% CI [{row['ci_low']:>8.4f}, {row['ci_high']:>8.4f}]")

rss_fit = float(np.sum((y_arr - D @ beta_ols) ** 2))
resid_se = np.sqrt(rss_fit / (len(y_arr) - D.shape[1]))
print(f"\nresidual std error = {resid_se:.3f}   (we generated the noise with sd = {NOISE_SD})")

 intercept  coef=  53.1631  se= 4.0374  t=   13.17  p=4.69e-30  95% CI [ 45.2092,  61.1170]
       age  coef=  -0.8290  se= 0.0577  t=  -14.38  p=4.41e-34  95% CI [ -0.9426,  -0.7154]
  bedrooms  coef=   9.9675  se= 0.6907  t=   14.43  p=2.90e-34  95% CI [  8.6067,  11.3283]
      sqft  coef=   0.1488  se= 0.0013  t=  117.84  p=1.09e-211  95% CI [  0.1463,   0.1513]

residual std error = 15.340   (we generated the noise with sd = 15.0)


Every true coefficient should sit inside its 95% interval, and the residual standard error
should land near the noise level we injected.

Now the part that matters: **does the standard-error formula actually work?** We can check it
without any theory. Regenerate the dataset many times with fresh noise, refit, and look at how
much each coefficient really moves. The empirical spread should match the analytic standard
errors.

In [17]:
sim_rng = np.random.default_rng(7)
n_sims = 4000
D_full = add_intercept_column(records_to_matrix(records, feature_names))
beta_sims = np.empty((n_sims, D_full.shape[1]))

for s in range(n_sims):
    y_sim = D_full @ beta_true + sim_rng.normal(0, NOISE_SD, n)
    beta_sims[s], *_ = np.linalg.lstsq(D_full, y_sim, rcond=None)

analytic = np.array([r["std_err"] for r in ols_inference(records_to_matrix(records, feature_names),
                                                         np.asarray(targets), feature_names)])
empirical = beta_sims.std(axis=0, ddof=1)

print(pd.DataFrame({
    "analytic_se": analytic,
    "empirical_se": empirical,
    "ratio": empirical / analytic,
}, index=["intercept"] + feature_names))
print(f"\n{n_sims:,} refits on all {n} rows (the table above used the {len(X_train)} training rows,")
print("so the standard errors differ slightly between the two -- more data, tighter estimates).")
print("Ratios near 1.0 mean the closed-form standard errors are correct.")

           analytic_se  empirical_se   ratio
intercept       3.5430        3.4555  0.9753
age             0.0506        0.0498  0.9830
bedrooms        0.5998        0.6017  1.0030
sqft            0.0011        0.0011  0.9811

4,000 refits on all 300 rows (the table above used the 240 training rows,
so the standard errors differ slightly between the two -- more data, tighter estimates).
Ratios near 1.0 mean the closed-form standard errors are correct.


---
# Part 2 — The pandas version

## Step 12 — `DataFrame` in, `DataFrame` out

The bridge is unchanged: `pd.DataFrame(records)` one way, `df.to_dict("records")` the other.
Two things carry over from the SGD notebook and matter just as much here:

- **reindex to the fitted column order** before `.to_numpy()`, or you silently score the wrong
  feature against the wrong coefficient;
- **preserve the index** on output, so predictions join back onto the source rows.

The reward for using pandas here is that the Step 11 statistics become a readable table.

In [18]:
df_train = pd.DataFrame(X_train)
df_test  = pd.DataFrame(X_test, index=[f"house_{i}" for i in test_i])
s_train  = pd.Series(y_train, name="price_k")

print(df_train.head(3))
print("\ncolumns:", list(df_train.columns), "<- insertion order")
print("fitted feature order:", ols.feature_names_, "<- sorted; they differ, which is the whole point")


def frame_to_matrix(df, feature_names):
    if not isinstance(df, pd.DataFrame):
        raise TypeError(f"expected a DataFrame, got {type(df).__name__}")
    missing = [c for c in feature_names if c not in df.columns]
    if missing:
        raise ValueError(f"DataFrame is missing column(s) {missing}; expected {feature_names}")
    unknown = [c for c in df.columns if c not in feature_names]
    if unknown:
        raise ValueError(f"DataFrame has unexpected column(s) {unknown}; fitted on {feature_names}")
    M = df[feature_names].to_numpy(dtype=float)      # reindex, THEN convert
    if not np.isfinite(M).all():
        raise ValueError(f"non-finite values in column(s) {df.columns[df.isna().any()].tolist()}")
    return M


correct = frame_to_matrix(df_test, ols.feature_names_) @ ols.coef_ + ols.intercept_
naive   = df_test.to_numpy(dtype=float) @ ols.coef_ + ols.intercept_


def r2(pred, y):
    y = np.asarray(y, dtype=float)
    return 1 - np.sum((y - pred) ** 2) / np.sum((y - y.mean()) ** 2)


print(f"\nR^2 reindexed  : {r2(correct, y_test):+.4f}")
print(f"R^2 .to_numpy(): {r2(naive,   y_test):+.4f}   <- wrong, and nothing raised")

     sqft  bedrooms   age
0   912.0         4  51.1
1   848.0         1  45.1
2  3297.0         3  51.8

columns: ['sqft', 'bedrooms', 'age'] <- insertion order
fitted feature order: ['age', 'bedrooms', 'sqft'] <- sorted; they differ, which is the whole point

R^2 reindexed  : +0.9869
R^2 .to_numpy(): -306.6773   <- wrong, and nothing raised


---
## Step 13 — One core, two adapters

Identical structure to the SGD notebook: the algorithm knows only about matrices, and thin
adapters handle the I/O format.

```
        _OLSCore
       /        \
LinearRegressionDictIO   LinearRegressionFrame
```

In [19]:
class _OLSCore:
    def __init__(self, fit_intercept=True, solver="lstsq"):
        self.fit_intercept = fit_intercept
        self.solver = solver

    def _feature_names(self, X):      raise NotImplementedError
    def _matrix(self, X):             raise NotImplementedError
    def _wrap_output(self, X, preds): raise NotImplementedError

    def fit(self, X, y):
        self.feature_names_in_ = self._feature_names(X)
        Xm = self._matrix(X)
        yv = np.asarray(y, dtype=float).ravel()
        if len(Xm) != len(yv):
            raise ValueError(f"X has {len(Xm)} rows but y has {len(yv)} targets")
        self.n_features_in_ = Xm.shape[1]

        x_off = Xm.mean(axis=0) if self.fit_intercept else np.zeros(Xm.shape[1])
        y_off = yv.mean() if self.fit_intercept else 0.0
        Xc, yc = Xm - x_off, yv - y_off

        self.rank_ = int(np.linalg.matrix_rank(Xc))
        self.singular_ = np.linalg.svd(Xc, compute_uv=False)
        if self.rank_ < Xc.shape[1] and self.solver != "lstsq":
            raise np.linalg.LinAlgError(
                f"rank deficient ({self.rank_} < {Xc.shape[1]}); use solver='lstsq'")

        if self.solver == "lstsq":
            self.coef_, *_ = np.linalg.lstsq(Xc, yc, rcond=None)
        elif self.solver == "solve":
            self.coef_ = np.linalg.solve(Xc.T @ Xc, Xc.T @ yc)
        else:
            self.coef_ = np.linalg.inv(Xc.T @ Xc) @ Xc.T @ yc

        self.intercept_ = float(y_off - self.coef_ @ x_off)
        return self

    def predict(self, X):
        if not hasattr(self, "coef_"):
            raise RuntimeError("call fit() before predict()")
        return self._wrap_output(X, self._matrix(X) @ self.coef_ + self.intercept_)

    @property
    def coef_dict_(self):
        return dict(zip(self.feature_names_in_, self.coef_))


class LinearRegressionDictIO(_OLSCore):
    def _feature_names(self, X):      return extract_feature_names(X)
    def _matrix(self, X):             return records_to_matrix(X, self.feature_names_in_)
    def _wrap_output(self, X, preds): return [{"prediction": float(p)} for p in preds]


class LinearRegressionFrame(_OLSCore):
    def _feature_names(self, X):
        if not isinstance(X, pd.DataFrame):
            raise TypeError(f"expected a DataFrame, got {type(X).__name__}")
        return list(X.columns)

    def _matrix(self, X):             return frame_to_matrix(X, self.feature_names_in_)
    def _wrap_output(self, X, preds): return pd.DataFrame({"prediction": preds}, index=X.index)

    def score(self, X, y):
        yv = np.asarray(y, dtype=float)
        return r2(self.predict(X)["prediction"].to_numpy(), yv)

    def summary_frame(self, X, y, alpha=0.05):
        rows = ols_inference(self._matrix(X), np.asarray(y, dtype=float),
                             self.feature_names_in_, alpha=alpha)
        return pd.DataFrame(rows).set_index("term")


m_dict  = LinearRegressionDictIO().fit(X_train, y_train)
m_frame = LinearRegressionFrame().fit(df_train, s_train)

print("dict  coef:", {k: round(float(v), 6) for k, v in m_dict.coef_dict_.items()})
print("frame coef:", {k: round(float(v), 6) for k, v in m_frame.coef_dict_.items()})
print("identical model despite different column order:",
      np.allclose(sorted(m_dict.coef_), sorted(m_frame.coef_)))

dict  coef: {'age': -0.828952, 'bedrooms': 9.967536, 'sqft': 0.148836}
frame coef: {'sqft': 0.148836, 'bedrooms': 9.967536, 'age': -0.828952}
identical model despite different column order: True


In [20]:
out_dict  = m_dict.predict(X_test)
out_frame = m_frame.predict(df_test)

print("--- list[dict] output ---")
print([{k: round(v, 2) for k, v in d.items()} for d in out_dict[:3]])

print("\n--- DataFrame output (index preserved) ---")
print(out_frame.head(3))
print(f"\nR^2 test: {m_frame.score(df_test, y_test):.4f}")

print("\n--- results joined back onto the source rows ---")
print(df_test.join(out_frame).assign(actual=y_test,
                                     residual=lambda d: d["actual"] - d["prediction"]).head())

--- list[dict] output ---
[{'prediction': 287.16}, {'prediction': 234.08}, {'prediction': 268.67}]

--- DataFrame output (index preserved) ---
           prediction
house_107    287.1583
house_90     234.0847
house_15     268.6718

R^2 test: 0.9869

--- results joined back onto the source rows ---
             sqft  bedrooms   age  prediction    actual  residual
house_107  1535.0         3  29.4    287.1583  284.9658   -2.1924
house_90   1211.0         5  59.3    234.0847  231.1788   -2.9059
house_15   1414.0         4  42.0    268.6718  270.8718    2.2000
house_286   933.0         1  12.4    191.7159  200.0812    8.3653
house_153  2296.0         5  43.2    408.9182  410.7850    1.8668


In [21]:
# The payoff for pandas: Step 11's statistics as a table.
summary = m_frame.summary_frame(df_train, s_train)
summary.insert(0, "true", [TRUE_INTERCEPT] + [TRUE_COEF[f] for f in m_frame.feature_names_in_])
summary["in_95_CI"] = (summary["true"] >= summary["ci_low"]) & (summary["true"] <= summary["ci_high"])
print(summary)

            true     coef  std_err         t      p_value   ci_low  ci_high  \
term                                                                          
intercept  50.00  53.1631   4.0374   13.1677   4.6850e-30  45.2092  61.1170   
sqft        0.15   0.1488   0.0013  117.8361  1.0886e-211   0.1463   0.1513   
bedrooms   10.00   9.9675   0.6907   14.4304   2.9021e-34   8.6067  11.3283   
age        -0.80  -0.8290   0.0577  -14.3761   4.4095e-34  -0.9426  -0.7154   

           in_95_CI  
term                 
intercept      True  
sqft           True  
bedrooms       True  
age            True  


---
## Step 14 — OLS vs SGD, measured

The usual claim is "OLS for small problems, SGD for big ones". Rather than repeat it, measure it.

The cost of OLS is roughly $O(n d^2 + d^3)$: it grows **linearly in rows** but **cubically in
features**. SGD is $O(n d)$ per epoch, so it degrades gently in both, and it never needs the
whole dataset in memory at once.

In [22]:
import time, warnings
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import SGDRegressor

SGD_MAX_ITER = 50   # capped deliberately -- see the note under the table

def timeit(fn, repeats=3):
    best = float("inf")
    for _ in range(repeats):
        t0 = time.perf_counter()
        out = fn()
        best = min(best, time.perf_counter() - t0)
    return best, out


bench_rng = np.random.default_rng(0)
rows = []

for n_rows, n_feats in [(1_000, 10), (100_000, 10), (5_000, 300), (5_000, 1_500)]:
    Xb = bench_rng.normal(size=(n_rows, n_feats))
    beta_b = bench_rng.normal(size=n_feats)
    yb = Xb @ beta_b + bench_rng.normal(0, 1.0, n_rows)
    Xb_s = (Xb - Xb.mean(0)) / Xb.std(0)          # SGD needs this; OLS does not

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", ConvergenceWarning)
        t_ols, m_ols = timeit(lambda: LinearRegression().fit(Xb, yb))
        t_sgd, m_sgd = timeit(lambda: SGDRegressor(max_iter=SGD_MAX_ITER, tol=1e-3,
                                                   random_state=0).fit(Xb_s, yb))

    rows.append({
        "n_rows": n_rows, "n_features": n_feats,
        "ols_sec": round(t_ols, 4), "sgd_sec": round(t_sgd, 4),
        "ols_R2": round(m_ols.score(Xb, yb), 4), "sgd_R2": round(m_sgd.score(Xb_s, yb), 4),
        "sgd_converged": m_sgd.n_iter_ < SGD_MAX_ITER,
        "faster": "OLS" if t_ols < t_sgd else "SGD",
    })

print(pd.DataFrame(rows).to_string(index=False))

 n_rows  n_features  ols_sec  sgd_sec  ols_R2  sgd_R2  sgd_converged faster
   1000          10   0.0010   0.0019  0.9387  0.9387           True    OLS
 100000          10   0.0245   0.0801  0.8963  0.8963           True    OLS
   5000         300   0.1010   0.0983  0.9965  0.9964           True    SGD
   5000        1500   2.1524   1.0571  0.9995  0.9993          False    SGD


Read that table honestly, because it does **not** say what the folklore says.

- **OLS wins on rows.** At 100,000 rows and 10 features it is still several times faster. The
  $n$ term in $O(nd^2 + d^3)$ is linear and BLAS handles it extremely well.
- **SGD wins on columns.** The crossover is in the feature count, not the row count. By 1,500
  features the $d^3$ term dominates and OLS is the slower one.
- **The SGD timings are flattered.** `sgd_converged` is `False` in the wide cases -- it hit the
  `max_iter` cap rather than converging, and its $R^2$ is correspondingly a shade lower. Part of
  its speed advantage is simply that it stopped early.

So "use SGD for big data" is really about **memory and streaming**, not wall-clock speed at these
sizes. OLS needs every row resident to form $X^{\top}X$ and cannot be updated when new data
arrives; SGD can consume data in batches forever via `partial_fit`. That, plus very wide feature
spaces, is the honest case for it.

---
## Recap

**The algorithm** — one line that matters:

```python
beta, *_ = np.linalg.lstsq(X_centered, y_centered, rcond=None)
intercept = y.mean() - beta @ X.mean(axis=0)
```

**The traps**

| Trap | Symptom | Fix |
|---|---|---|
| `inv(X.T @ X)` | confident nonsense, no exception | `np.linalg.lstsq(X, y)` |
| Collinear features | unstable, arbitrary coefficients | check `rank_`; drop or combine features |
| Reading coefficients as "importance" | misleading when features are correlated or on different scales | standardise first, and report intervals |
| DataFrame column order | wrong predictions, no error | reindex to `feature_names_in_` |
| Trusting `std_err` blindly | invalid intervals under heteroscedasticity or dependence | check the four assumptions |
| Lower training RSS than the truth | overfitting, by construction | evaluate on held-out data |

**Against the SGD notebook**

| | SGD | OLS |
|---|---|---|
| Standardisation | mandatory (else `nan`) | optional, changes nothing |
| Hyper-parameters | many, and they matter | none |
| Reproducibility | depends on seed and shuffle | exact |
| Uncertainty estimates | not available in closed form | standard errors, t, CIs |
| Rank deficiency | quietly absorbed | detectable via `rank_` |
| Streaming / `partial_fit` | yes | no — needs all data at once |

**Exercises**

1. Add `sample_weight` to `fit`. The weighted normal equations are
   $X^{\top}WX\boldsymbol\beta = X^{\top}W\mathbf{y}$; the `lstsq` trick is to scale both $X$ and
   $\mathbf{y}$ rows by $\sqrt{w_i}$. Verify against sklearn's `sample_weight`.
2. Implement Ridge by appending $\sqrt{\alpha}\,I$ to the centred design matrix and zeros to
   $\mathbf{y}$, then calling the same `lstsq`. Confirm it now handles the Step 6 duplicate
   column without any special case.
3. Compute a variance inflation factor per feature by regressing each feature on the others.
   What does it report for `sqft` and `sqft_m2` in Step 6?
4. `summary_frame` recomputes the fit internally. Refactor it to reuse the fitted `coef_`, and
   confirm the standard errors are unchanged.
5. Predict the log of price instead of price. Does the residual standard error still recover
   `NOISE_SD`, and why not?